In [ ]:
import numpy as np
from scipy.stats import pearsonr
from pathlib import Path
from mintpy.utils import utils as ut
from mintpy.cli import view, generate_mask, subset, reference_point, mask as mintpy_mask

In [ ]:
timeseries_dir = Path('/Users/havazli/Desktop/SnowWaterEquivalent/ALOS2_CA_P67_F750_A/CA_P67_F750_A_Dec22_June23/mintpy/geo').expanduser()
timeseries_file = timeseries_dir.joinpath('geo_timeseries_ERA5.h5')
lidar_file = Path('resampled_ASO_Truckee_2023Apr09_snowdepth_3m-006.tif')

## Generate mask from LIDAR data

In [ ]:
#Visualize LIDAR file
view.main([
    str(lidar_file),
    '--sub-lat', '38.68', '39.65',
    '--sub-lon', '-119.78', '-120.50',
    # '-v', '0', '5',
    '--noverbose',
])

In [ ]:
outname_lidar_mask = 'lidar_mask.h5'
mask_opts = [str(lidar_file),
             '-o', outname_lidar_mask,
            ]
# Run generate mask
generate_mask.main(mask_opts)

#Visualize the mask
view.main([
    outname_lidar_mask,
    '--noverbose',
])

## Mask the timeseries

In [ ]:
masked_timeseries = timeseries_file.with_name(timeseries_file.stem + ('_msk') + timeseries_file.suffix)
mintpy_mask.main([
    str(timeseries_file),
    '--mask', outname_lidar_mask,
    '--outfile', str(masked_timeseries),
])

In [ ]:
view.main([
    str(masked_timeseries), '20230406',
    '--sub-lat', '38.68', '39.65',
    '--sub-lon', '-119.78', '-120.50',
    '-v', '-10', '10',
    # '--noreference',
    '--noverbose',
])

In [ ]:
view.main([
    str(lidar_file),
    '--sub-lat', '38.68', '39.65',
    '--sub-lon', '-119.78', '-120.50',
    '-v', '0', '5',
    '--noreference',
    '--noverbose',
])

## Reference Timeseries

In [ ]:
# ref_point_opts = (str(masked_timeseries),
#                   '--lat', '39.50',
#                   '--lon', '-120.50')
# reference_point.main(ref_point_opts)

## Subset Data

In [ ]:
# Subset data and write to new files
subset_masked_timeseries = masked_timeseries.with_name(masked_timeseries.stem + ('_sub') + masked_timeseries.suffix)
subset.main([str(masked_timeseries),
             '--lat', '38.68', '39.65',
             '--lon', '-119.78', '-120.50',
             '-o', str(subset_masked_timeseries)
])

In [ ]:
subset_lidar = lidar_file.with_name(lidar_file.stem + ('_sub.h5')) 
subset.main([str(lidar_file),
             '--lat', '38.68', '39.65',
             '--lon', '-119.78', '-120.50',
             '-o', str(subset_lidar)
])

## Read timeseries data set and calculate correlation

In [ ]:
ts_data, _ = ut.readfile.read(subset_masked_timeseries, datasetName='timeseries-20230406')
lidar_data, _ = ut.readfile.read(subset_lidar)

In [ ]:
nan_mask = ~np.isnan(lidar_data) & ~np.isnan(ts_data)
lidar_data_masked = lidar_data[nan_mask]
ts_data_masked = ts_data[nan_mask].ravel()*-100 # Multiplying by NEGATIVE 100 to convert to cm
corr = pearsonr(lidar_data_masked, ts_data_masked)
print(corr)

In [ ]:
ts_data_all, atr = ut.readfile.read(subset_masked_timeseries, datasetName='timeseries')
dates = ut.readfile.get_slice_list(subset_masked_timeseries)

In [ ]:
dates

In [ ]:
ts_data_sum_to_april = ts_data_all[0:9].sum(axis=0)

In [ ]:
nan_mask = ~np.isnan(lidar_data) & ~np.isnan(ts_data_sum_to_april)
lidar_data_masked = lidar_data[nan_mask]
ts_data_masked = ts_data_sum_to_april[nan_mask].ravel()*-100
corr = pearsonr(lidar_data_masked, ts_data_masked)
print(corr)

In [ ]:
for d in dates:
    ts_data, _ = ut.readfile.read(subset_masked_timeseries, datasetName=d)
    nan_mask = ~np.isnan(lidar_data) & ~np.isnan(ts_data)
    lidar_data_masked = lidar_data[nan_mask]
    ts_data_masked = ts_data[nan_mask].ravel()*-100 # Multiplying by NEGATIVE 100 to change direction and convert to cm
    corr = pearsonr(lidar_data_masked, ts_data_masked)
    print(d, corr)